In [ ]:
# Created on Sept 25, 2025
# Created by: Afeena, Sara, Bishr, Muhammed
# Purpose: This file will be used to read from our slices (user-severity, env-severity, time-severity, user-time-env-severity) 
# and make summaries.

import pandas as pd
import numpy as np
import os as os

# File Paths
YearlySummariesLocation = "C:\\Users\\afeen\\Documents\\MBAI\\Business Analytics\\FlaskProjects\\Master Code (BA)\\03Summaries\\YearlySummaries.csv"
UserSeverityLocation = "C:\\Users\\afeen\\Documents\\MBAI\\Business Analytics\\FlaskProjects\\Master Code (BA)\\03Summaries\\UserSeveritySlice.csv"
UserSummariesLocation = "C:\\Users\\afeen\\Documents\\MBAI\\Business Analytics\\FlaskProjects\\Master Code (BA)\\03Summaries\\UserSummaries.csv"
EnvSeverityLocation = "C:\\Users\\afeen\\Documents\\MBAI\\Business Analytics\\FlaskProjects\\Master Code (BA)\\03Summaries\\EnvSeveritySlice.csv"
EnvSummaryLocation = "C:\\Users\\afeen\\Documents\\MBAI\\Business Analytics\\FlaskProjects\\Master Code (BA)\\03Summaries\\EnvSummaries.csv"
TimeSeverityLocation = "C:\\Users\\afeen\\Documents\\MBAI\\Business Analytics\\FlaskProjects\\Master Code (BA)\\03Summaries\\TimeSeveritySlice.csv"
TimeSummaryLocation = "C:\\Users\\afeen\\Documents\\MBAI\\Business Analytics\\FlaskProjects\\Master Code (BA)\\03Summaries\\TimeSummaries.csv"

# Read the CSV file
df_userSeverity = pd.read_csv(UserSeverityLocation)
df_envSeverity = pd.read_csv(EnvSeverityLocation)
df_timeSeverity = pd.read_csv(TimeSeverityLocation)

# FUNCTION- TO PRESERVE THE SEVERITY LABELS
def expand_severity(row):
    if row["SeverityOfInjury_Fatal"] == 1:
        return "Fatal"
    elif row["SeverityOfInjury_Major"] == 1:
        return "Major"
    elif row["SeverityOfInjury_Minor"] == 1:
        return "Minor"
    else:
        return "Unknown"
    
# FUNCTION- TO PRESERVE THE SEVERITY LABELS    
def collapse_severity(row):
    if row["SeverityOfInjury_Fatal"] == 1 or row["SeverityOfInjury_Major"] == 1:
        return 1  # Severe
    else:
        return 0  # Not Severe
    
 # FUNCTION- TO CREATE CROSSTAB
 # SummaryType: "counts" or "proportions"
 # rowVar: variable to be displayed in rows
 # groupBy: variable to be displayed in columns   
def create_crosstab(whichDataFrame,summaryType,groupBy,rowVar2):
    if summaryType == "counts":
        return pd.crosstab(
            # whichDataFrame[rowVar],
             [whichDataFrame[var] for var in groupBy] if isinstance(groupBy, list) else whichDataFrame[groupBy],
            whichDataFrame[rowVar2]
        )
    elif summaryType == "proportions":
        return pd.crosstab(
            [whichDataFrame[var] for var in groupBy] if isinstance(groupBy, list) else whichDataFrame[groupBy],
            whichDataFrame[rowVar2],
            normalize="index"
        )

####################################################################################################################
########################## SEVERITY BY YEAR ############################################################
####################################################################################################################  
# 
# NO COLLAPSE OF SEVERITY
df_userSeverity["SeverityOfInjury"] = df_userSeverity.apply(expand_severity, axis=1)


# summary_year_severity = pd.crosstab(
#     df_userSeverity["SeverityOfInjury"],
#     df_userSeverity["YEAR_COLLISION_OCCURED"],
# )
# print("\n",summary_year_severity)
# summary_year_severity.to_csv(UserSummariesLocation, mode='w', header=True)  

countYearSev = create_crosstab(df_userSeverity,"counts","YEAR_COLLISION_OCCURED","SeverityOfInjury")

print("\n",countYearSev)
# countYearSev.to_csv(YearlySummariesLocation, mode='w', header=True)

propsYearSev = create_crosstab(df_userSeverity,"proportions","YEAR_COLLISION_OCCURED","SeverityOfInjury")
YearlySummaries = countYearSev.astype(str) + " (" + (propsYearSev*100).round(1).astype(str) + "%)"
print("\n",propsYearSev)
YearlySummaries.to_csv(YearlySummariesLocation, mode='w', header=True)

########################################################################################################################
########################################### USER-SEVERITY SUMMARY ###########################################
########################################################################################################################

df_userSeverity["SeverityOfInjury"] = df_userSeverity.apply(collapse_severity, axis=1)

countInvTypeSev = create_crosstab(df_userSeverity,"counts","InvolvementType","SeverityOfInjury")
propsInvTypeSev = create_crosstab(df_userSeverity,"proportions","InvolvementType","SeverityOfInjury")

summaryUserInvTypeSeverity = countInvTypeSev.astype(str) + " (" + (propsInvTypeSev*100).round(1).astype(str) + "%)"
summaryUserInvTypeSeverity.to_csv(UserSummariesLocation, mode='a', header=True)

###### (COLLAPSE-INVOLVEMENT TYPE) COLLAPSE SERVERITY INTO BINARY INDICATOR (SEVERE(FATAL, MAJOR) VS NOT SEVERE)
# df_userSeverity["SeverityBinary"] = (df_userSeverity["SeverityOfInjury_Fatal"] | df_userSeverity["SeverityOfInjury_Major"]).astype(int)
# Counts (raw numbers)
# counts = pd.crosstab(
#     df_userSeverity["InvolvementType"], 
#     df_userSeverity["SeverityBinary"]
# )

# Percentages by Row (Proportions)
# props = pd.crosstab(
#     df_userSeverity["InvolvementType"], 
#     df_userSeverity["SeverityBinary"], 
#     normalize="index"
# )

# Combine into one table (counts + percentages)
# summary_UserInvTypeSeverity = counts.astype(str) + " (" + (props*100).round(1).astype(str) + "%)"
# print(summary_UserInvTypeSeverity)
# summary_UserInvTypeSeverity.to_csv(UserSummariesLocation, mode='w', header=True)

## ---- End COLLAPSE-INVOLVEMENT TYPE ---- ##

###### (DO NOT COLLAPSE-INVOLVEMENT TYPE) KEEP SERVERITY AS: FATAL, MAJOR, MINOR
df_userSeverity["SeverityOfInjury"] = df_userSeverity.apply(expand_severity, axis=1)

countInvTypeSev = create_crosstab(df_userSeverity,"counts","InvolvementType","SeverityOfInjury")
propsInvTypeSev = create_crosstab(df_userSeverity,"proportions","InvolvementType","SeverityOfInjury")

summaryUserInvTypeSeverity = countInvTypeSev.astype(str) + " (" + (propsInvTypeSev*100).round(1).astype(str) + "%)"
summaryUserInvTypeSeverity.to_csv(UserSummariesLocation, mode='a', header=True)

# # Counts (raw numbers)
# summary_inv_severity = pd.crosstab(
#     df_userSeverity["InvolvementType"],
#     df_userSeverity["SeverityOfInjury"],
# )

# # print("\n",summary_age_severity)
# # summary_age_severity.to_csv(UserSummariesLocation, mode='a', header=True)

# # Percentages by Row (Proportions)
# propsInvType = pd.crosstab(
#     df_userSeverity["InvolvementType"], 
#     df_userSeverity["SeverityOfInjury"], 
#     margins=True,
#     normalize="index"  # proportions
# )

# # Combine into one table (raw numbers + percentages)
# propsInvType = summary_inv_severity.astype(str) + " (" + (propsInvType*100).round(1).astype(str) + "%)"
# print("\n",propsInvType)
# propsInvType.to_csv(UserSummariesLocation, mode='a', header=True)

## ---- End DO NOT COLLAPSE-INVOLVEMENT TYPE ---- ##

###### (COLLAPSE-AGE RANGE) COLLAPSE SERVERITY INTO BINARY INDICATOR (SEVERE(FATAL, MAJOR) VS NOT SEVERE)

df_userSeverity["SeverityOfInjury"] = df_userSeverity.apply(collapse_severity, axis=1)

countInvTypeSev = create_crosstab(df_userSeverity,"counts","AgeOfInvolvedParty","SeverityOfInjury")
propsInvTypeSev = create_crosstab(df_userSeverity,"proportions","AgeOfInvolvedParty","SeverityOfInjury")

summaryAgeRangeSeverity = countInvTypeSev.astype(str) + " (" + (propsInvTypeSev*100).round(1).astype(str) + "%)"
summaryAgeRangeSeverity.to_csv(UserSummariesLocation, mode='a', header=True)


# df_userSeverity["SeverityBinary"] = (df_userSeverity["SeverityOfInjury_Fatal"] | df_userSeverity["SeverityOfInjury_Major"]).astype(int)

# # Counts (raw numbers)
# counts = pd.crosstab(
#     df_userSeverity["AgeOfInvolvedParty"], 
#     df_userSeverity["SeverityBinary"]
# )

# # Percentages by Row (Proportions)
# props = pd.crosstab(
#     df_userSeverity["AgeOfInvolvedParty"], 
#     df_userSeverity["SeverityBinary"], 
#     normalize="index"
# )

# # Combine into one table (counts + percentages)
# summary_UserAgeRangeSeverity = counts.astype(str) + " (" + (props*100).round(1).astype(str) + "%)"
# print(summary_UserAgeRangeSeverity)
# summary_UserAgeRangeSeverity.to_csv(UserSummariesLocation, mode='a', header=True)

## ---- End COLLAPSE-AGE RANGE ---- ##
 
###### (DO NOT COLLAPSE-AGE RANGE) DO NOT COLLAPSE SERVERITY. KEEPT IT AS: FATAL, MAJOR, MINOR

df_userSeverity["SeverityOfInjury"] = df_userSeverity.apply(expand_severity, axis=1)

countAgeRangeSev = create_crosstab(df_userSeverity,"counts","AgeOfInvolvedParty","SeverityOfInjury")
propsAgeRangeSev = create_crosstab(df_userSeverity,"proportions","AgeOfInvolvedParty","SeverityOfInjury")

summaryAgeRangeSeverity = countAgeRangeSev.astype(str) + " (" + (propsAgeRangeSev*100).round(1).astype(str) + "%)"
summaryAgeRangeSeverity.to_csv(UserSummariesLocation, mode='a', header=True)


# df_userSeverity["SeverityOfInjury"] = df_userSeverity.apply(expand_severity, axis=1)

# # Counts (raw numbers)
# summary_age_severity = pd.crosstab(
#     df_userSeverity["AgeOfInvolvedParty"],
#     df_userSeverity["SeverityOfInjury"],
# )

# # print("\n",summary_age_severity)
# # summary_age_severity.to_csv(UserSummariesLocation, mode='a', header=True)

# # Percentages by Row (Proportions)
# propsAgeInvParty = pd.crosstab(
#     df_userSeverity["AgeOfInvolvedParty"], 
#     df_userSeverity["SeverityOfInjury"], 
#     margins=True,
#     normalize="index"  # proportions
# )

# # Combine into one table (raw numbers + percentages)
# propsAgeInvParty = summary_age_severity.astype(str) + " (" + (propsAgeInvParty*100).round(1).astype(str) + "%)"
# print("\n",propsAgeInvParty)
# propsAgeInvParty.to_csv(UserSummariesLocation, mode='a', header=True)

## ---- End DO NOT COLLAPSE-AGE RANGE ---- ##

##########################################
# DO NOT COLLAPSE (COMBINATION OF AGE RANGE & INVOLVEMENT TYPE): FATAL, MAJOR, MINOR
##########################################

df_userSeverity["SeverityOfInjury"] = df_userSeverity.apply(expand_severity, axis=1)

countUserComboSev = create_crosstab(df_userSeverity,"counts",["AgeOfInvolvedParty","InvolvementType"],"SeverityOfInjury")
propsUserComboSev = create_crosstab(df_userSeverity,"proportions",["AgeOfInvolvedParty","InvolvementType"],"SeverityOfInjury")

summaryAgeRangeSeverity = countUserComboSev.astype(str) + " (" + (propsUserComboSev*100).round(1).astype(str) + "%)"
summaryAgeRangeSeverity.to_csv(UserSummariesLocation, mode='a', header=True)

# df_userSeverity["SeverityOfInjury"] = df_userSeverity.apply(expand_severity, axis=1)

# # Lighting/RoadSurface by Severity
# usr_combo_counts = pd.crosstab(
#     [df_userSeverity["AgeOfInvolvedParty"], df_userSeverity["InvolvementType"]],
#     df_userSeverity["SeverityOfInjury"]
# )

# print("\nRaw counts:\n")
# print(usr_combo_counts)
    
# # Convert to row percentages (proportions per condition)
# usr_combo_props = pd.crosstab(
#     [df_userSeverity["AgeOfInvolvedParty"], df_userSeverity["InvolvementType"]],
#     df_userSeverity["SeverityOfInjury"],
#     normalize="index"
# )

# print("\nProportions (%):")
# print(usr_combo_props)

# # Optional: combine counts and percentages into one table
# usr_combo_summary = usr_combo_counts.astype(str) + " (" + (usr_combo_props*100).round(1).astype(str) + "%)"
# print("\nCombined summary:")
# print(usr_combo_summary)


# # Save to CSV if needed
# usr_combo_summary.to_csv(UserSummariesLocation, mode='a', header=True)

##### ---- End DO NOT COLLAPSE (COMBINATION OF AGE RANGE & INVOLVEMENT TYPE) ---- ##


df_userSeverity["SeverityOfInjury"] = df_userSeverity.apply(collapse_severity, axis=1)

countAgeInvTypeComboSev = create_crosstab(df_userSeverity,"counts",["AgeOfInvolvedParty","InvolvementType"],"SeverityOfInjury")
propsAgeInvTypeComboSev = create_crosstab(df_userSeverity,"proportions",["AgeOfInvolvedParty","InvolvementType"],"SeverityOfInjury")

summaryAgeRangeSeverity = countAgeInvTypeComboSev.astype(str) + " (" + (propsAgeInvTypeComboSev*100).round(1).astype(str) + "%)"
summaryAgeRangeSeverity.to_csv(UserSummariesLocation, mode='a', header=True)

####################################################################################################################
########################## ENVIRONMENT SEVERITY SUMMARY ############################################################
####################################################################################################################

#############################
# COLLAPSE (FOR LIGHT CONDITION) SERVERITY INTO BINARY INDICATOR (SEVERE(FATAL, MAJOR) VS NOT SEVERE)
#############################

df_envSeverity["SeverityOfInjury"] = df_envSeverity.apply(collapse_severity, axis=1)

countLightSev = create_crosstab(df_envSeverity,"counts","LightingCondition","SeverityOfInjury")
propsLightSev = create_crosstab(df_envSeverity,"proportions","LightingCondition","SeverityOfInjury")

summaryLightSeverity = countLightSev.astype(str) + " (" + (propsLightSev*100).round(1).astype(str) + "%)"
summaryLightSeverity.to_csv(EnvSummaryLocation, mode='a', header=True)

# df_envSeverity["SeverityBinary"] = (df_envSeverity["SeverityOfInjury_Fatal"] | df_envSeverity["SeverityOfInjury_Major"]).astype(int)

# # Counts (raw numbers)
# LightCounts = pd.crosstab(
#     df_envSeverity["LightingCondition"], 
#     df_envSeverity["SeverityBinary"]
# )

# # Percentages by Row (Proportions)
# LightProps = pd.crosstab(
#     df_envSeverity["LightingCondition"], 
#     df_envSeverity["SeverityBinary"], 
#     normalize="index"
# )

# # Combine into one table (counts + percentages)
# summary_EnvLightSeverity = LightCounts.astype(str) + " (" + (LightProps*100).round(1).astype(str) + "%)"
# print(summary_EnvLightSeverity)
# summary_EnvLightSeverity.to_csv(EnvSummaryLocation, mode='w', header=True)

#############################
# DO NOT COLLAPSE (FOR LIGHT CONDITION): FATAL, MAJOR, MINOR
#############################

df_envSeverity["SeverityOfInjury"] = df_envSeverity.apply(expand_severity, axis=1)

countLightSev = create_crosstab(df_envSeverity,"counts","LightingCondition","SeverityOfInjury")
propsLightSev = create_crosstab(df_envSeverity,"proportions","LightingCondition","SeverityOfInjury")

summaryLightSeverity = countLightSev.astype(str) + " (" + (propsLightSev*100).round(1).astype(str) + "%)"
summaryLightSeverity.to_csv(EnvSummaryLocation, mode='a', header=True)

# df_envSeverity["SeverityOfInjury"] = df_envSeverity.apply(expand_severity, axis=1) # Call function to preserve severity labels

# # Counts (raw numbers)
# summary_light_severity = pd.crosstab(
#     df_envSeverity["LightingCondition"],
#     df_envSeverity["SeverityOfInjury"],
# )

# # print("\n",summary_age_severity)
# # summary_age_severity.to_csv(UserSummariesLocation, mode='a', header=True)

# # Percentages by Row (Proportions)
# LightCondProps = pd.crosstab(
#     df_envSeverity["LightingCondition"], 
#     df_envSeverity["SeverityOfInjury"], 
#     margins=True,
#     normalize="index"  # proportions
# )

# # Combine into one table (raw numbers + percentages)
# propsLightCondition = summary_light_severity.astype(str) + " (" + (LightCondProps*100).round(1).astype(str) + "%)"
# print("\n",propsLightCondition)
# propsLightCondition.to_csv(EnvSummaryLocation, mode='a', header=True)


#############################
# COLLAPSE (FOR ROAD CONDITION) SERVERITY INTO BINARY INDICATOR (SEVERE(FATAL, MAJOR) VS NOT SEVERE)
#############################

df_envSeverity["SeverityOfInjury"] = df_envSeverity.apply(collapse_severity, axis=1)

countRoadSev = create_crosstab(df_envSeverity,"counts","RoadSurfaceCondition","SeverityOfInjury")
propsRoadSev = create_crosstab(df_envSeverity,"proportions","RoadSurfaceCondition","SeverityOfInjury")

summaryRoadSeverity = countRoadSev.astype(str) + " (" + (propsRoadSev*100).round(1).astype(str) + "%)"
summaryRoadSeverity.to_csv(EnvSummaryLocation, mode='a', header=True)


# # SEVERITY BY ROAD CONDITION
# # Counts (raw numbers)
# RoadCounts = pd.crosstab(
#     df_envSeverity["RoadSurfaceCondition"], 
#     df_envSeverity["SeverityBinary"]
# )

# # Percentages by Row (Proportions)
# RoadProps = pd.crosstab(
#     df_envSeverity["RoadSurfaceCondition"], 
#     df_envSeverity["SeverityBinary"], 
#     normalize="index"
# )

# # Combine into one table (counts + percentages)
# summary_EnvRoadSeverity = RoadCounts.astype(str) + " (" + (RoadProps*100).round(1).astype(str) + "%)"
# print(summary_EnvRoadSeverity)
# summary_EnvRoadSeverity.to_csv(EnvSummaryLocation, mode='a', header=True)

#############################
# DO NOT COLLAPSE (FOR ROAD CONDITION): FATAL, MAJOR, MINOR
#############################

df_envSeverity["SeverityOfInjury"] = df_envSeverity.apply(expand_severity, axis=1)

countRoadSev = create_crosstab(df_envSeverity,"counts","RoadSurfaceCondition","SeverityOfInjury")
propsRoadSev = create_crosstab(df_envSeverity,"proportions","RoadSurfaceCondition","SeverityOfInjury")

summaryRoadSeverity = countRoadSev.astype(str) + " (" + (propsRoadSev*100).round(1).astype(str) + "%)"
summaryRoadSeverity.to_csv(EnvSummaryLocation, mode='a', header=True)


# df_envSeverity["SeverityOfInjury"] = df_envSeverity.apply(collapse_severity, axis=1) # Call function to preserve severity labels

# # Counts (raw numbers)
# summary_road_severity = pd.crosstab(
#     df_envSeverity["RoadSurfaceCondition"],
#     df_envSeverity["SeverityOfInjury"],
# )

# # print("\n",summary_age_severity)
# # summary_age_severity.to_csv(UserSummariesLocation, mode='a', header=True)

# # Percentages by Row (Proportions)
# RoadCondProps = pd.crosstab(
#     df_envSeverity["RoadSurfaceCondition"], 
#     df_envSeverity["SeverityOfInjury"], 
#     margins=True,
#     normalize="index"  # proportions
# )

# # Combine into one table (raw numbers + percentages)
# propsRoadCondition = summary_road_severity.astype(str) + " (" + (RoadCondProps*100).round(1).astype(str) + "%)"
# print("\n",propsRoadCondition)
# propsRoadCondition.to_csv(EnvSummaryLocation, mode='a', header=True)

##########################################
# COMBINATION OF LIGHT CONDITION & ROAD SURFACE CONDITION: COLLAPSE SEVERITY
##########################################

df_envSeverity["SeverityOfInjury"] = df_envSeverity.apply(collapse_severity, axis=1)

countComboRoadLightSev = create_crosstab(df_envSeverity,"counts",["RoadSurfaceCondition","LightingCondition"],"SeverityOfInjury")
propsComboRoadLightSev = create_crosstab(df_envSeverity,"proportions",["RoadSurfaceCondition","LightingCondition"],"SeverityOfInjury")

summaryComboRoadLightSeverity = countComboRoadLightSev.astype(str) + " (" + (propsComboRoadLightSev*100).round(1).astype(str) + "%)"
summaryComboRoadLightSeverity.to_csv(EnvSummaryLocation, mode='a', header=True)
##########################################
# COMBINATION OF LIGHT CONDITION & ROAD SURFACE CONDITION: DO NOT COLLAPSE SEVERITY
##########################################


df_envSeverity["SeverityOfInjury"] = df_envSeverity.apply(expand_severity, axis=1)

countComboRoadLightSev = create_crosstab(df_envSeverity,"counts",["RoadSurfaceCondition","LightingCondition"],"SeverityOfInjury")
propsComboRoadLightSev = create_crosstab(df_envSeverity,"proportions",["RoadSurfaceCondition","LightingCondition"],"SeverityOfInjury")

summaryComboRoadLightSeverity = countComboRoadLightSev.astype(str) + " (" + (propsComboRoadLightSev*100).round(1).astype(str) + "%)"
summaryComboRoadLightSeverity.to_csv(EnvSummaryLocation, mode='a', header=True)


# df_envSeverity["SeverityOfInjury"] = df_envSeverity.apply(expand_severity, axis=1)

# # Lighting/RoadSurface by Severity
# env_combo_counts = pd.crosstab(
#     [df_envSeverity["LightingCondition"], df_envSeverity["RoadSurfaceCondition"]],
#     df_envSeverity["SeverityOfInjury"]
# )

# print("\nRaw counts:\n")
# print(env_combo_counts)

# # Convert to row percentages (proportions per condition)
# env_combo_props = pd.crosstab(
#     [df_envSeverity["LightingCondition"], df_envSeverity["RoadSurfaceCondition"]],
#     df_envSeverity["SeverityOfInjury"],
#     normalize="index"
# ).round(3) * 100  # convert to percentages

# print("\nProportions (%):")
# print(env_combo_props)

# # Optional: combine counts and percentages into one table
# env_combo_summary = env_combo_counts.astype(str) + " (" + env_combo_props.astype(str) + "%)"
# print("\nCombined summary:")
# print(env_combo_summary)

# # Save to CSV if needed
# env_combo_summary.to_csv(EnvSummaryLocation, mode='a', header=True)


###########################
# TIME SEVERITY SUMMARY
###########################


# Time Severity Summary



# count the number of collisions by PEAK HOUR and severity type
df_timeSeverity["SeverityOfInjury"] = df_timeSeverity.apply(collapse_severity, axis=1)

countPeakSev = create_crosstab(df_timeSeverity,"counts","PEAKOFFPEAK","SeverityOfInjury")
propsPeakSev = create_crosstab(df_timeSeverity,"proportions","PEAKOFFPEAK","SeverityOfInjury")

summaryPeakSeverity = countPeakSev.astype(str) + " (" + (propsPeakSev*100).round(1).astype(str) + "%)"
summaryPeakSeverity.to_csv(TimeSummaryLocation, mode='w', header=True)

df_timeSeverity["SeverityOfInjury"] = df_timeSeverity.apply(expand_severity, axis=1)

countPeakSev = create_crosstab(df_timeSeverity,"counts","PEAKOFFPEAK","SeverityOfInjury")
propsPeakSev = create_crosstab(df_timeSeverity,"proportions","PEAKOFFPEAK","SeverityOfInjury")

summaryPeakSeverity = countPeakSev.astype(str) + " (" + (propsPeakSev*100).round(1).astype(str) + "%)"
summaryPeakSeverity.to_csv(TimeSummaryLocation, mode='a', header=True)

#count the number of collisions by TIME RANGE and severity type

df_timeSeverity["SeverityOfInjury"] = df_timeSeverity.apply(collapse_severity, axis=1)

countTimeRangeSev = create_crosstab(df_timeSeverity,"counts","TIME_RANGE","SeverityOfInjury")
propsTimeRangeSev = create_crosstab(df_timeSeverity,"proportions","TIME_RANGE","SeverityOfInjury")

summaryTimeRangeSeverity = countTimeRangeSev.astype(str) + " (" + (propsTimeRangeSev*100).round(1).astype(str) + "%)"
summaryTimeRangeSeverity.to_csv(TimeSummaryLocation, mode='a', header=True)

df_timeSeverity["SeverityOfInjury"] = df_timeSeverity.apply(expand_severity, axis=1)

countTimeRangeSev = create_crosstab(df_timeSeverity,"counts","TIME_RANGE","SeverityOfInjury")
propsTimeRangeSev = create_crosstab(df_timeSeverity,"proportions","TIME_RANGE","SeverityOfInjury")

summaryTimeRangeSeverity = countTimeRangeSev.astype(str) + " (" + (propsTimeRangeSev*100).round(1).astype(str) + "%)"
summaryTimeRangeSeverity.to_csv(TimeSummaryLocation, mode='a', header=True)

#count the number of collisions by HOUR and severity type

df_timeSeverity["SeverityOfInjury"] = df_timeSeverity.apply(collapse_severity, axis=1)

countHourSev = create_crosstab(df_timeSeverity,"counts","HOUR_COLLISION_OCCURED","SeverityOfInjury")
propsHourSev = create_crosstab(df_timeSeverity,"proportions","HOUR_COLLISION_OCCURED","SeverityOfInjury")

summaryHourSeverity = countHourSev.astype(str) + " (" + (propsHourSev*100).round(1).astype(str) + "%)"
summaryHourSeverity.to_csv(TimeSummaryLocation, mode='a', header=True)

df_timeSeverity["SeverityOfInjury"] = df_timeSeverity.apply(expand_severity, axis=1)

countHourSev = create_crosstab(df_timeSeverity,"counts","HOUR_COLLISION_OCCURED","SeverityOfInjury")
propsHourSev = create_crosstab(df_timeSeverity,"proportions","HOUR_COLLISION_OCCURED","SeverityOfInjury")

summaryHourSeverity = countHourSev.astype(str) + " (" + (propsHourSev*100).round(1).astype(str) + "%)"
summaryHourSeverity.to_csv(TimeSummaryLocation, mode='a', header=True)

# count the number of collisions by MONTH NAME and severity type
df_timeSeverity["SeverityOfInjury"] = df_timeSeverity.apply(collapse_severity, axis=1)

countMonthNamerSev = create_crosstab(df_timeSeverity,"counts","MONTH_NAME_COLLISION_OCCURED","SeverityOfInjury")
propsMonthNameSev = create_crosstab(df_timeSeverity,"proportions","MONTH_NAME_COLLISION_OCCURED","SeverityOfInjury")

summaryMonthNameSeverity = countMonthNamerSev.astype(str) + " (" + (propsMonthNameSev*100).round(1).astype(str) + "%)"
summaryMonthNameSeverity.to_csv(TimeSummaryLocation, mode='a', header=True)

df_timeSeverity["SeverityOfInjury"] = df_timeSeverity.apply(expand_severity, axis=1)

countMonthNamerSev = create_crosstab(df_timeSeverity,"counts","MONTH_NAME_COLLISION_OCCURED","SeverityOfInjury")
propsMonthNameSev = create_crosstab(df_timeSeverity,"proportions","MONTH_NAME_COLLISION_OCCURED","SeverityOfInjury")

summaryMonthNameSeverity = countMonthNamerSev.astype(str) + " (" + (propsMonthNameSev*100).round(1).astype(str) + "%)"
summaryMonthNameSeverity.to_csv(TimeSummaryLocation, mode='a', header=True)

# count the number of collisions by WINTER - NON WINTER and severity type

df_timeSeverity["SeverityOfInjury"] = df_timeSeverity.apply(collapse_severity, axis=1)

countSeasonSev = create_crosstab(df_timeSeverity,"counts","WINTER_NOTWINTER","SeverityOfInjury")
propsSeasonSev = create_crosstab(df_timeSeverity,"proportions","WINTER_NOTWINTER","SeverityOfInjury")

summarySeasonSeverity = countSeasonSev.astype(str) + " (" + (propsSeasonSev*100).round(1).astype(str) + "%)"
summarySeasonSeverity.to_csv(TimeSummaryLocation, mode='a', header=True)

df_timeSeverity["SeverityOfInjury"] = df_timeSeverity.apply(expand_severity, axis=1)

countSeasonSev = create_crosstab(df_timeSeverity,"counts","WINTER_NOTWINTER","SeverityOfInjury")
propsSeasonSev = create_crosstab(df_timeSeverity,"proportions","WINTER_NOTWINTER","SeverityOfInjury")

summarySeasonSeverity = countSeasonSev.astype(str) + " (" + (propsSeasonSev*100).round(1).astype(str) + "%)"
summarySeasonSeverity.to_csv(TimeSummaryLocation, mode='a', header=True)

#month and hour combination
df_timeSeverity["SeverityOfInjury"] = df_timeSeverity.apply(collapse_severity, axis=1)

countMonthHourSev = create_crosstab(df_timeSeverity,"counts",["MONTH_NAME_COLLISION_OCCURED","HOUR_COLLISION_OCCURED"],"SeverityOfInjury")
propsMonthHourSev = create_crosstab(df_timeSeverity,"proportions",["MONTH_NAME_COLLISION_OCCURED","HOUR_COLLISION_OCCURED"],"SeverityOfInjury")

summaryMonthHourSeverity = countMonthHourSev.astype(str) + " (" + (propsMonthHourSev*100).round(1).astype(str) + "%)"
summaryMonthHourSeverity.to_csv(TimeSummaryLocation, mode='a', header=True)

df_timeSeverity["SeverityOfInjury"] = df_timeSeverity.apply(expand_severity, axis=1)

countMonthHourSev = create_crosstab(df_timeSeverity,"counts",["MONTH_NAME_COLLISION_OCCURED","HOUR_COLLISION_OCCURED"],"SeverityOfInjury")
propsMonthHourSev = create_crosstab(df_timeSeverity,"proportions",["MONTH_NAME_COLLISION_OCCURED","HOUR_COLLISION_OCCURED"],"SeverityOfInjury")

summaryMonthHourSeverity = countMonthHourSev.astype(str) + " (" + (propsMonthHourSev*100).round(1).astype(str) + "%)"
summaryMonthHourSeverity.to_csv(TimeSummaryLocation, mode='a', header=True)



print("Summaries Created Successfully")


 SeverityOfInjury        Fatal  Major  Minor  Unknown
YEAR_COLLISION_OCCURED                              
2006                       57    488    121       85
2007                       51    442    122      103
2008                       54    394    104       71
2009                       48    431     91       82
2010                       43    402     84       73
2011                       34    392     87       60
2012                       44    430     92       78
2013                       63    401    110       60
2014                       51    330     66       67
2015                       65    319     68       65
2016                       78    337     96       69
2017                       61    352     62       83
2018                       60    382    100       50
2019                       62    330     62       65
2020                       40    246     43       33
2021                       60    241     55       32
2022                       49    264     64 